In [331]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Part 7 – Loading Data into Staging

**Sources loaded in this section:**
- `streaming_service.csv` → staging for `DIM_SUBSCRIPTION_PLAN` and `FACT_SUBSCRIPTION_PRICING`
- `platform_summary.csv` → staging for `DIM_PLATFORM`

In [334]:
StreamingService_df = pd.read_csv('Data Sources copy/streaming_service.csv')
StreamingService_df.head()

,service,date,price
0,Netflix,Jul-2011,7.99
1,Netflix,Aug-2011,7.99
2,Netflix,Sep-2011,7.99
3,Netflix,Oct-2011,7.99
4,Netflix,Nov-2011,7.99


In [336]:
PlatformSummary_df = pd.read_csv('Data Sources copy/Streaming Platform Dataset/platform_summary.csv')
PlatformSummary_df.head()

,platform,launch_year,parent_company,content_focus,business_model,reporting_date,subscribers_millions,quarterly_revenue_usd_millions,quarterly_profit_usd_millions,quarterly_eps_usd,...,projected_2026_subscribers_millions,sports_rights_spend_2026_usd_millions,sports_rights_global_share_pct,streaming_revenue_q4_usd_millions,streaming_revenue_growth_pct,monthly_streams_millions,streams_growth_pct,monthly_hours_watched_millions,monthly_hours_growth_pct,peak_concurrent_viewers_millions
0,Netflix,2007,Netflix Inc.,"Original series, films, licensed content",Subscription + Ads,2026-01-19,325.0,12050.0,2410.0,0.56,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Disney+,2019,The Walt Disney Company,"Family, Marvel, Star Wars, National Geographic",Subscription + Ads,2026-02-07,126.0,NaN,NaN,NaN,...,284.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Hulu,2008,The Walt Disney Company,"TV shows, next-day broadcast",Subscription + Ads,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Amazon Prime Video,2006,Amazon.com Inc.,"Original series, movies, sports",Prime bundle + Subscription,2026-02-06,NaN,NaN,NaN,NaN,...,NaN,3800.0,27.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Paramount+,2021,Paramount Global,"Nickelodeon, CBS, Showtime",Subscription + Ads,2026-02-06,NaN,NaN,NaN,NaN,...,NaN,1136.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [338]:
print("StreamingService_df shape:", StreamingService_df.shape)
print("PlatformSummary_df shape:", PlatformSummary_df.shape)

StreamingService_df shape: (777, 3)
PlatformSummary_df shape: (12, 36)


In [340]:
print("Record Count of StreamingService_df:", len(StreamingService_df))
print("Record Count of PlatformSummary_df:", len(PlatformSummary_df))

Record Count of StreamingService_df: 777
Record Count of PlatformSummary_df: 12


In [342]:
# print("StreamingService_df counts:", StreamingService_df.count())
# print("\nPlatformSummary_df counts:", PlatformSummary_df.count())

# Part 8 – Transformations

#### Duplicate values checking StreamingService_df

In [346]:
StreamingServiceDistinct_df.isnull().sum()

service                       0
date                          0
price                         0
price_tier                    0
price_change_mom             10
cumulative_price_increase     0
platform_name                 0
dtype: int64

In [348]:
StreamingServiceDistinct_df = StreamingService_df.drop_duplicates()

In [350]:
#no duplicates
print('Rows of original:', StreamingService_df.shape[0])
print('Rows of distinct df:', StreamingServiceDistinct_df.shape[0])
print(f'\nRecords removed: {StreamingService_df.shape[0] - StreamingServiceDistinct_df.shape[0]}')

Rows of original: 777
Rows of distinct df: 777

Records removed: 0


#### Transformation 1: Date Standardization

In [353]:
print('Before transformation:')
print(StreamingServiceDistinct_df[['service', 'date', 'price']].head(5).to_string())

Before transformation:
   service      date  price
0  Netflix  Jul-2011   7.99
1  Netflix  Aug-2011   7.99
2  Netflix  Sep-2011   7.99
3  Netflix  Oct-2011   7.99
4  Netflix  Nov-2011   7.99


In [355]:
StreamingServiceDistinct_df['date'] = pd.to_datetime(
    StreamingServiceDistinct_df['date'], format='%b-%Y'
)

print('\nAfter transformation')
print(StreamingServiceDistinct_df[['service', 'date', 'price']].head(5).to_string())


After transformation
   service       date  price
0  Netflix 2011-07-01   7.99
1  Netflix 2011-08-01   7.99
2  Netflix 2011-09-01   7.99
3  Netflix 2011-10-01   7.99
4  Netflix 2011-11-01   7.99


#### Transformation 2 – Price Tiers

In [358]:
print('Before transformation')
print(StreamingServiceDistinct_df[['service', 'price']].describe())

Before transformation
            price
count  777.000000
mean     9.378674
std      2.609859
min      4.990000
25%      7.990000
50%      8.990000
75%      9.990000
max     17.990000


In [360]:
bins   = [0, 7, 14, float('inf')]
labels = ['Budget ($0-$7)', 'Mid ($7-$14)', 'Premium (>$14)']
StreamingServiceDistinct_df['price_tier'] = pd.cut(
    StreamingServiceDistinct_df['price'], bins=bins, labels=labels, right=True
)
print('\nAfter transformation')
print(StreamingServiceDistinct_df[['service', 'date', 'price', 'price_tier']].head(10).to_string())

print('\nTier distribution')
print(StreamingServiceDistinct_df['price_tier'].value_counts())


After transformation
   service       date  price    price_tier
0  Netflix 2011-07-01   7.99  Mid ($7-$14)
1  Netflix 2011-08-01   7.99  Mid ($7-$14)
2  Netflix 2011-09-01   7.99  Mid ($7-$14)
3  Netflix 2011-10-01   7.99  Mid ($7-$14)
4  Netflix 2011-11-01   7.99  Mid ($7-$14)
5  Netflix 2011-12-01   7.99  Mid ($7-$14)
6  Netflix 2012-01-01   7.99  Mid ($7-$14)
7  Netflix 2012-02-01   7.99  Mid ($7-$14)
8  Netflix 2012-03-01   7.99  Mid ($7-$14)
9  Netflix 2012-04-01   7.99  Mid ($7-$14)

Tier distribution
price_tier
Mid ($7-$14)      560
Budget ($0-$7)    154
Premium (>$14)     63
Name: count, dtype: int64


#### Transformation 3: price_change_mom

In [363]:
StreamingServiceDistinct_df = StreamingServiceDistinct_df.sort_values(['service', 'date'])

StreamingServiceDistinct_df['price_change_mom'] = (
    StreamingServiceDistinct_df.groupby('service')['price'].diff()
)

print('Months where price changed:')
price_changes = StreamingServiceDistinct_df[StreamingServiceDistinct_df['price_change_mom'] != 0].dropna(subset=['price_change_mom'])
print(price_changes[['service', 'date', 'price', 'price_change_mom']].to_string())

Months where price changed:
         service       date  price  price_change_mom
588    Apple TV+ 2022-10-01   6.99               2.0
603    Apple TV+ 2024-01-01   9.99               3.0
167      Disney+ 2021-03-01   7.99               1.0
188      Disney+ 2022-12-01  10.99               3.0
198      Disney+ 2023-10-01  13.99               3.0
235      HBO Max 2023-02-01  15.99               1.0
319         Hulu 2021-10-01  12.99               1.0
333         Hulu 2022-12-01  14.99               2.0
346         Hulu 2024-01-01  17.99               3.0
90       Netflix 2019-01-01   8.99               1.0
126      Netflix 2022-01-01   9.99               1.0
147      Netflix 2023-10-01  15.49               5.5
451   Paramount+ 2023-06-01  11.99               2.0
641      Peacock 2023-08-01  11.99               2.0
552  Prime Video 2024-01-01  11.99               3.0
729      Shudder 2023-08-01   6.99               1.0


#### Transformation 4 – Cumulative Price Increase

In [366]:
StreamingServiceDistinct_df['cumulative_price_increase'] = (
    StreamingServiceDistinct_df.groupby('service')['price'].transform(lambda x: x - x.iloc[0])
)

print('Cumulative price increase per service (latest month):')
latest = (StreamingServiceDistinct_df.sort_values('date').groupby('service').last().reset_index())
print(latest[['service', 'date', 'price', 'cumulative_price_increase']].to_string())

Cumulative price increase per service (latest month):
       service       date  price  cumulative_price_increase
0    Apple TV+ 2024-01-01   9.99                        5.0
1  Crunchyroll 2024-01-01   7.99                        0.0
2      Disney+ 2024-01-01  13.99                        7.0
3      HBO Max 2024-01-01  15.99                        1.0
4         Hulu 2024-01-01  17.99                        6.0
5      Netflix 2024-01-01  15.49                        7.5
6   Paramount+ 2024-01-01  11.99                        2.0
7      Peacock 2024-01-01  11.99                        2.0
8  Prime Video 2024-01-01  11.99                        3.0
9      Shudder 2024-01-01   6.99                        1.0


#### PlatformSummary_df

In [369]:
PlatformSummary_df[dim_cols].isnull().sum()

platform          0
parent_company    0
content_focus     0
business_model    0
launch_year       0
dtype: int64

In [371]:
PlatformSummaryDistinct_df = PlatformSummary_df[['platform', 'parent_company', 'content_focus', 'business_model', 'launch_year']].drop_duplicates()

In [373]:
print('Rows of original:', PlatformSummary_df.shape[0])
print('Rows of distinct df:', PlatformSummaryDistinct_df.shape[0])
print(f'\nRecords removed: {PlatformSummary_df.shape[0] - PlatformSummaryDistinct_df.shape[0]}')

Rows of original: 12
Rows of distinct df: 12

Records removed: 0


#### Transformation 5: Derived Columns for DIM_PLATFORM

In [376]:
print('Before transformation')
display(PlatformSummaryDistinct_df)

Before transformation


,platform,parent_company,content_focus,business_model,launch_year
0,Netflix,Netflix Inc.,"Original series, films, licensed content",Subscription + Ads,2007
1,Disney+,The Walt Disney Company,"Family, Marvel, Star Wars, National Geographic",Subscription + Ads,2019
2,Hulu,The Walt Disney Company,"TV shows, next-day broadcast",Subscription + Ads,2008
3,Amazon Prime Video,Amazon.com Inc.,"Original series, movies, sports",Prime bundle + Subscription,2006
4,Paramount+,Paramount Global,"Nickelodeon, CBS, Showtime",Subscription + Ads,2021
5,Peacock,Comcast,NBCUniversal content,Subscription + Ads,2020
6,Apple TV+,Apple Inc.,"Original series, films",Subscription,2019
7,Max,Warner Bros. Discovery,"HBO, Discovery, Warner Bros.",Subscription + Ads,2020
8,Roku,Roku Inc.,"FAST platform, advertising",AVOD + Transactional,2008
9,YouTube TV,Google,"User-generated, sports",AVOD + Subscription,2017


In [378]:
# all platforms in this source are digital streaming services
PlatformSummaryDistinct_df['is_digital'] = True
PlatformSummaryDistinct_df['media_sector'] = 'Streaming'
PlatformSummaryDistinct_df['platform_age_years'] = 2026 - PlatformSummaryDistinct_df['launch_year'].astype(int)
PlatformSummaryDistinct_df['launch_decade'] = (PlatformSummaryDistinct_df['launch_year'].astype(int) // 10 * 10).astype(str) + 's'


print('\nAfter transformation')
display(PlatformSummaryDistinct_df[['platform', 'launch_year', 'launch_decade', 'is_digital', 'media_sector', 'platform_age_years']])


After transformation


,platform,launch_year,launch_decade,is_digital,media_sector,platform_age_years
0,Netflix,2007,2000s,True,Streaming,19
1,Disney+,2019,2010s,True,Streaming,7
2,Hulu,2008,2000s,True,Streaming,18
3,Amazon Prime Video,2006,2000s,True,Streaming,20
4,Paramount+,2021,2020s,True,Streaming,5
5,Peacock,2020,2020s,True,Streaming,6
6,Apple TV+,2019,2010s,True,Streaming,7
7,Max,2020,2020s,True,Streaming,6
8,Roku,2008,2000s,True,Streaming,18
9,YouTube TV,2017,2010s,True,Streaming,9


#### Transformation 6 – Service Name Mapping

In [381]:
services_in_pricing  = set(StreamingServiceDistinct_df['service'].unique())
platforms_in_summary = set(PlatformSummaryDistinct_df['platform'].unique())

print('In pricing but NOT in platform_summary (need to add):')
print(sorted(services_in_pricing - platforms_in_summary))

print('\nIn platform_summary but NOT in pricing (no pricing history):')
print(sorted(platforms_in_summary - services_in_pricing))

In pricing but NOT in platform_summary (need to add):
['Crunchyroll', 'HBO Max', 'Prime Video', 'Shudder']

In platform_summary but NOT in pricing (no pricing history):
['Amazon Prime Video', 'Max', 'Pluto TV', 'Roku', 'Tubi', 'YouTube TV']


In [383]:
name_map = {'HBO Max': 'Max', 'Prime Video' : 'Amazon Prime Video'}
StreamingServiceDistinct_df['platform_name'] = StreamingServiceDistinct_df['service'].replace(name_map)

print('Service → platform_name mapping applied:')
print(StreamingServiceDistinct_df[['service', 'platform_name']].drop_duplicates().to_string())

Service → platform_name mapping applied:
         service       platform_name
553    Apple TV+           Apple TV+
735  Crunchyroll         Crunchyroll
151      Disney+             Disney+
202      HBO Max                 Max
247         Hulu                Hulu
0        Netflix             Netflix
347   Paramount+          Paramount+
604      Peacock             Peacock
459  Prime Video  Amazon Prime Video
647      Shudder             Shudder


# Part 9: Loading

#### DIM_PLATFORM

In [393]:
PlatformMerge_df = pd.merge(StreamingServiceDistinct_df[['platform_name']].drop_duplicates(), PlatformSummaryDistinct_df,
    left_on='platform_name', right_on='platform', how='left', indicator=True
)
display(PlatformMerge_df)

,platform_name,platform,parent_company,content_focus,business_model,launch_year,is_digital,media_sector,platform_age_years,launch_decade,_merge
0,Apple TV+,Apple TV+,Apple Inc.,"Original series, films",Subscription,2019.0,True,Streaming,7.0,2010s,both
1,Crunchyroll,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,Disney+,Disney+,The Walt Disney Company,"Family, Marvel, Star Wars, National Geographic",Subscription + Ads,2019.0,True,Streaming,7.0,2010s,both
3,Max,Max,Warner Bros. Discovery,"HBO, Discovery, Warner Bros.",Subscription + Ads,2020.0,True,Streaming,6.0,2020s,both
4,Hulu,Hulu,The Walt Disney Company,"TV shows, next-day broadcast",Subscription + Ads,2008.0,True,Streaming,18.0,2000s,both
5,Netflix,Netflix,Netflix Inc.,"Original series, films, licensed content",Subscription + Ads,2007.0,True,Streaming,19.0,2000s,both
6,Paramount+,Paramount+,Paramount Global,"Nickelodeon, CBS, Showtime",Subscription + Ads,2021.0,True,Streaming,5.0,2020s,both
7,Peacock,Peacock,Comcast,NBCUniversal content,Subscription + Ads,2020.0,True,Streaming,6.0,2020s,both
8,Amazon Prime Video,Amazon Prime Video,Amazon.com Inc.,"Original series, movies, sports",Prime bundle + Subscription,2006.0,True,Streaming,20.0,2000s,both
9,Shudder,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [395]:
PlatformMerge_df['_merge'].value_counts()

_merge
both          8
left_only     2
right_only    0
Name: count, dtype: int64

In [397]:
AddPlatforms_df = PlatformMerge_df[PlatformMerge_df['_merge'] == 'left_only'][['platform_name']]
print('Platforms to add to dimension table:')
print(AddPlatforms_df)

Platforms to add to dimension table:
  platform_name
1   Crunchyroll
9       Shudder


In [401]:
#building using metadata
NewPlatformRows_df = pd.DataFrame([
    {'platform': 'Crunchyroll', 'parent_company': 'Sony Group Corporation',
     'content_focus': 'Anime, manga',     'business_model': 'Subscription',
     'launch_year': 2009, 'is_digital': True, 'media_sector': 'Streaming',
     'platform_age_years': 2026 - 2009, 'launch_decade': '2000s'},
    {'platform': 'Shudder',     'parent_company': 'AMC Networks',
     'content_focus': 'Horror, thriller', 'business_model': 'Subscription',
     'launch_year': 2015, 'is_digital': True, 'media_sector': 'Streaming',
     'platform_age_years': 2026 - 2015, 'launch_decade': '2010s'},
])
print('New platform rows:')
display(NewPlatformRows_df)

New platform rows:


,platform,parent_company,content_focus,business_model,launch_year,is_digital,media_sector,platform_age_years,launch_decade
0,Crunchyroll,Sony Group Corporation,"Anime, manga",Subscription,2009,True,Streaming,17,2000s
1,Shudder,AMC Networks,"Horror, thriller",Subscription,2015,True,Streaming,11,2010s


In [403]:
#adding together
DimPlatform = pd.concat([PlatformSummaryDistinct_df, NewPlatformRows_df], ignore_index=True)

In [405]:
print('Rows of original:', PlatformSummaryDistinct_df.shape[0])
print('Rows after concat:', DimPlatform.shape[0])
print(f'\nRecords added: {DimPlatform.shape[0] - PlatformSummaryDistinct_df.shape[0]}')

Rows of original: 12
Rows after concat: 14

Records added: 2


In [407]:
# SCD Type 2 attributes – initial load: all records are current
DimPlatform['effective_date'] = pd.to_datetime(DimPlatform['launch_year'].astype(str) + '-01-01')
DimPlatform['end_date'] = None
DimPlatform['is_current'] = True

In [409]:
#surrogate key
DimPlatform['platform_key'] = DimPlatform.reset_index().index + 1

print('\nDimPlatform with surrogate key:')
display(DimPlatform[['platform_key','platform','parent_company','media_sector',
                      'business_model','launch_year','launch_decade','is_digital',
                      'platform_age_years','effective_date','end_date','is_current']])


DimPlatform with surrogate key:


,platform_key,platform,parent_company,media_sector,business_model,launch_year,launch_decade,is_digital,platform_age_years,effective_date,end_date,is_current
0,1,Netflix,Netflix Inc.,Streaming,Subscription + Ads,2007,2000s,True,19,2007-01-01,None,True
1,2,Disney+,The Walt Disney Company,Streaming,Subscription + Ads,2019,2010s,True,7,2019-01-01,None,True
2,3,Hulu,The Walt Disney Company,Streaming,Subscription + Ads,2008,2000s,True,18,2008-01-01,None,True
3,4,Amazon Prime Video,Amazon.com Inc.,Streaming,Prime bundle + Subscription,2006,2000s,True,20,2006-01-01,None,True
4,5,Paramount+,Paramount Global,Streaming,Subscription + Ads,2021,2020s,True,5,2021-01-01,None,True
5,6,Peacock,Comcast,Streaming,Subscription + Ads,2020,2020s,True,6,2020-01-01,None,True
6,7,Apple TV+,Apple Inc.,Streaming,Subscription,2019,2010s,True,7,2019-01-01,None,True
7,8,Max,Warner Bros. Discovery,Streaming,Subscription + Ads,2020,2020s,True,6,2020-01-01,None,True
8,9,Roku,Roku Inc.,Streaming,AVOD + Transactional,2008,2000s,True,18,2008-01-01,None,True
9,10,YouTube TV,Google,Streaming,AVOD + Subscription,2017,2010s,True,9,2017-01-01,None,True


In [411]:
print('Rows and columns of DimPlatform:', DimPlatform.shape)
print('Record Count of DimPlatform:', len(DimPlatform))
print('DimPlatform counts:', DimPlatform[['platform_key','platform','is_current']].count())

Rows and columns of DimPlatform: (14, 13)
Record Count of DimPlatform: 14
DimPlatform counts: platform_key    14
platform        14
is_current      14
dtype: int64
